In [1]:
# Importar librerías
import pandas as pd
import os
from src.ingestion import *

In [3]:
# Generar lista de tickers del fichero constituents del S&P 500
data_folder = "data"
os.makedirs(data_folder, exist_ok=True) # Crear carpeta si no existe
df_tickers = pd.read_csv(f"{data_folder}/constituents.csv")

# Modificar "BRK.B" a "BRK-B" y "BF.B" a "BF-B" para evitar problemas con yfinance
df_tickers["Symbol"] = df_tickers["Symbol"].replace("BRK.B", "BRK-B")
df_tickers["Symbol"] = df_tickers["Symbol"].replace("BF.B", "BF-B")

tickers_list = df_tickers["Symbol"].tolist()
len(tickers_list)

502

In [4]:
# Seleccionar y renombrar columnas en df_tickers
df_tickers = df_tickers[["Symbol", "GICS Sector","GICS Sub-Industry", "Date added"]]
df_tickers.rename(columns={
    "Symbol": "Ticker",
    "GICS Sector": "Sector",
    "GICS Sub-Industry": "SubIndustry", 
    "Date added": "DateAdded"
    }, inplace=True)
df_tickers.head()

,Ticker,Sector,SubIndustry,DateAdded
0,MMM,Industrials,Industrial Conglomerates,1957-03-04
1,AOS,Industrials,Building Products,2017-07-26
2,ABT,Health Care,Health Care Equipment,1957-03-04
3,ABBV,Health Care,Biotechnology,2012-12-31
4,ACN,Information Technology,IT Consulting & Other Services,2011-07-06


In [5]:
# Eliminar espacios en los nombres de los sectores y sub-industrias
df_tickers["Sector"] = df_tickers["Sector"].str.replace(" ", "")
df_tickers["SubIndustry"] = df_tickers["SubIndustry"].str.replace(" ", "")

In [6]:
# Extraer precios de los tickers y del índice SPY (demora unos 3 minutos)
df_prices = extraer_precios(tickers_list)
df_index = extraer_precios(['SPY'])

In [7]:
# Calcular los retornos mensuales, varianza del activo y covarianza con el mercado para cada ticker
df_retornos = calcular_retornos(df_prices, df_index)
df_retornos.tail()

,Fecha,Ticker,Retorno_Mensual,Varianza_Activo,Covarianza_Mercado
19427,2026-02-01,ZTS,0.054797,0.003722,0.000187
19428,2026-03-01,ZTS,-0.098322,0.004268,0.000685
19429,2026-04-01,ZTS,-0.027409,0.004203,0.000590
19430,2026-05-01,ZTS,-0.321319,0.010122,-0.000605
19431,2026-06-01,ZTS,0.024585,0.010600,-0.000851


In [ ]:
# Extraer datos del Balance y estado de Resultados (demora 7 minutos)
df_fundamentals = extraer_datos_fundamentales(tickers_list)
df_fundamentals.head()

In [ ]:
df_merged = pd.merge(df_prices, df_fundamentals, on=['Ticker', 'Fecha'], how='outer')

# --- LA CORRECCIÓN DEL LOOKAHEAD BIAS ---
# Desplazamos la fecha del reporte 60 días hacia el futuro para simular 
# el momento en que el dato se vuelve realmente público para los inversores.
retraso_dias = 60
df_fundamentals['Fecha'] = df_fundamentals['Fecha'] + pd.DateOffset(days=retraso_dias)
# ----------------------------------------

# 2. Hacer el OUTER merge
df_merged = pd.merge(df_prices, df_fundamentals, on=['Ticker', 'Fecha'], how='outer')

# 3. Ordenar cronológicamente (CRUCIAL para el ffill)
df_merged = df_merged.sort_values(by=['Ticker', 'Fecha'])

# 4. Aplicar Forward Fill
# Identificamos qué columnas vienen de los estados financieros (las que tienen NaN)
cols_financieras = ['Total Revenue', 'Gross Profit', 'Operating Income', 'Net Income', 
                    'EBITDA', 'Basic Average Shares', 'Cash And Cash Equivalents', 'Current Debt', 
                    'Long Term Debt', 'Total Debt', 'Stockholders Equity',
                    'Total Assets', 'Current Assets', 'Current Liabilities'
                    ]

# Aplicamos el ffill() EXCLUSIVAMENTE a esas columnas, agrupando por Ticker
df_merged[cols_financieras] = df_merged.groupby('Ticker')[cols_financieras].ffill()

# 5. Limpiar las filas huérfanas creadas por el outer merge
df_merged_clean = df_merged.dropna(subset=['Close'])

# 6. ELIMINAR LOS NaN INICIALES 
# Filtramos por una columna fundamental crítica. Las filas anteriores al primer
# reporte financiero disponible serán eliminadas de forma limpia.
columna_critica = 'EBITDA' # O 'Net Income', la que uses sí o sí en tus ratios
df_merged_clean = df_merged_clean.dropna(subset=[columna_critica])

# Resetear el índice para que quede estético de 0 a N
df_merged_clean = df_merged_clean.reset_index(drop=True)

# Unir los betas calculados al dataframe limpio final
df_merged_clean = pd.merge(df_merged_clean, df_retornos, on=['Ticker', 'Fecha'], how='left')

df_merged_clean.info()

       Fecha       Close Ticker  Total Revenue  Gross Profit  \
0 2023-03-01  134.998413      A   6.848000e+09  3.722000e+09   
1 2023-04-01  132.158707      A   6.848000e+09  3.722000e+09   
2 2023-05-01  113.059898      A   6.848000e+09  3.722000e+09   
3 2023-06-01  117.536552      A   6.848000e+09  3.722000e+09   
4 2023-07-01  119.249535      A   6.848000e+09  3.722000e+09   

   Operating Income    Net Income        EBITDA  Basic Average Shares  \
0      1.618000e+09  1.254000e+09  1.905000e+09           299000000.0   
1      1.618000e+09  1.254000e+09  1.905000e+09           299000000.0   
2      1.618000e+09  1.254000e+09  1.905000e+09           299000000.0   
3      1.618000e+09  1.254000e+09  1.905000e+09           299000000.0   
4      1.618000e+09  1.254000e+09  1.905000e+09           299000000.0   

   Cash And Cash Equivalents  Current Debt  Long Term Debt    Total Debt  \
0               1.053000e+09    36000000.0    2.733000e+09  2.769000e+09   
1               1.053000

In [13]:
df_with_metrics = calcular_metricas(df_merged_clean)

# Luego de calcular los ratios, se eliminan las columnas financieras 
df_with_metrics = df_with_metrics.drop(columns=cols_financieras)
df_with_metrics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17501 entries, 0 to 17500
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Fecha               17501 non-null  datetime64[ns]
 1   Close               17501 non-null  float64       
 2   Ticker              17501 non-null  object        
 3   Retorno_Mensual     17149 non-null  float64       
 4   Varianza_Activo     17149 non-null  float64       
 5   Covarianza_Mercado  17149 non-null  float64       
 6   MarketCap           17445 non-null  float64       
 7   EnterpriseValue     17445 non-null  float64       
 8   PE_Trailing         17407 non-null  float64       
 9   EnterpriseToEbitda  17445 non-null  float64       
 10  PriceToBook         17397 non-null  float64       
 11  operatingMargins    17501 non-null  float64       
 12  profitMargins       17463 non-null  float64       
 13  returnOnEquity      17415 non-null  float64   

In [14]:
# Unir df_with_metrics con df_tickers
df_final = pd.merge(df_with_metrics, df_tickers, on="Ticker", how="left")
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17501 entries, 0 to 17500
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Fecha               17501 non-null  datetime64[ns]
 1   Close               17501 non-null  float64       
 2   Ticker              17501 non-null  object        
 3   Retorno_Mensual     17149 non-null  float64       
 4   Varianza_Activo     17149 non-null  float64       
 5   Covarianza_Mercado  17149 non-null  float64       
 6   MarketCap           17445 non-null  float64       
 7   EnterpriseValue     17445 non-null  float64       
 8   PE_Trailing         17407 non-null  float64       
 9   EnterpriseToEbitda  17445 non-null  float64       
 10  PriceToBook         17397 non-null  float64       
 11  operatingMargins    17501 non-null  float64       
 12  profitMargins       17463 non-null  float64       
 13  returnOnEquity      17415 non-null  float64   

In [16]:
# Extraer datos macro (opcional, devuelve dataframe vacio si no se configura la clave API de FRED)
indicadores = ['FEDFUNDS', 'GS10', 'T10Y2Y', 'CPIAUCSL', 'UNRATE']
df_datos_macro = extraer_datos_macro(indicadores)
df_datos_macro.head()

c:\Users\sebas\OneDrive\Documentos\python-development\FinDataMining\src\ingestion.py:338: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_macro = df_macro.resample('M').last()


,Fecha,FEDFUNDS,GS10,T10Y2Y,CPIAUCSL,UNRATE
0,2022-06-30,1.21,3.14,0.06,294.957,3.6
1,2022-07-31,1.68,2.90,-0.22,294.913,3.5
2,2022-08-31,2.33,2.90,-0.30,295.097,3.6
3,2022-09-30,2.56,3.52,-0.39,296.349,3.5
4,2022-10-31,3.08,3.98,-0.41,298.007,3.6


In [18]:
# Asegurar que la columna sea tipo datetime
df_datos_macro['Fecha'] = pd.to_datetime(df_datos_macro['Fecha'])

# 2. Desplazar la fecha al 1er día del mes, saltando un mes de publicación para evitar fuga de datos
# Ej: 2022-06-30 -> 2022-08-01
df_datos_macro['Fecha'] = df_datos_macro['Fecha'] + pd.offsets.MonthBegin(2)

In [19]:
# Se une con los datos de df_final
df_raw = pd.merge(df_final, df_datos_macro, on="Fecha", how= "left")
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17501 entries, 0 to 17500
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Fecha               17501 non-null  datetime64[ns]
 1   Close               17501 non-null  float64       
 2   Ticker              17501 non-null  object        
 3   Retorno_Mensual     17149 non-null  float64       
 4   Varianza_Activo     17149 non-null  float64       
 5   Covarianza_Mercado  17149 non-null  float64       
 6   MarketCap           17445 non-null  float64       
 7   EnterpriseValue     17445 non-null  float64       
 8   PE_Trailing         17407 non-null  float64       
 9   EnterpriseToEbitda  17445 non-null  float64       
 10  PriceToBook         17397 non-null  float64       
 11  operatingMargins    17501 non-null  float64       
 12  profitMargins       17463 non-null  float64       
 13  returnOnEquity      17415 non-null  float64   

In [20]:
# Guardar datos extraidos en fichero raw_data
df_raw.to_parquet(f"{data_folder}/raw_data.parquet")